# Distillation : un élève rapide qui imite le zero-shot de la thèse

**Objectif.** Le zero-shot `MoritzLaurer/deberta-v3-large-zeroshot-v2.0` (l'**enseignant**) lit chaque verbatim **28 fois**, une fois par émotion. C'est ce qui coûte 1 à 2 minutes par verbatim sur CPU. Ce notebook entraîne un **élève** qui lit le verbatim **une seule fois** et sort directement :

1. les **28 probabilités** d'émotions, chacune évaluée indépendamment (comme le zero-shot en multi-label) ;
2. les **5 scores factoriels de Bartlett** de la thèse, calculés à partir de ces 28 probabilités par une couche mathématique **figée**.

```
ENTRAÎNEMENT (GPU, une fois)                      PRODUCTION (CPU, chaque nouveau verbatim)
160 000 verbatims ─► élève ─► 28 probas           verbatim ─► élève (1 lecture) ─► 28 probas
                        │                                                        │
   comparées aux 28 probas du zero-shot                            couche figée de la thèse
                        │                                                        │
              correction de l'élève                                     5 scores de Bartlett
```

**Ce que l'élève peut et ne peut pas faire.** Il apprend à **reproduire** le zero-shot, avec ses qualités et ses défauts. Toutes les métriques de ce notebook mesurent donc la **fidélité à l'enseignant**, pas la « vérité » émotionnelle. C'est exactement l'objectif : une imitation rapide des résultats de la thèse.

**Plan du notebook**

| Partie | Ce qu'on fait | Où ça tourne |
|---|---|---|
| 1 | Configuration | — |
| 2 | Chargement et jointure des données | CPU |
| 3 | Reconstruction de la couche factorielle de la thèse | CPU |
| 4 | Découpage entraînement / validation / test | CPU |
| 5 | Tokenisation et longueur des textes | CPU |
| 6 | Construction de l'élève | GPU |
| 7 | Entraînement | GPU (≈ 1 à quelques heures) |
| 8 | Évaluation finale sur le test | GPU |
| 9 | Export pour le CPU (ONNX + int8) et mesure de vitesse | CPU |
| 10 | Fonction de prédiction pour la production | CPU |

**Prérequis** (dans l'environnement du projet, pas celui de vLLM) :
```bash
uv add transformers onnx onnxruntime scikit-learn matplotlib tqdm sentencepiece protobuf
```

## 1. Configuration

Tous les réglages sont ici, pour pouvoir relancer une expérience en ne changeant qu'une cellule.

**Le choix du GPU se fait avant d'importer `torch`.** `CUDA_VISIBLE_DEVICES` ne rend visible que le GPU choisi : pour PyTorch, il s'appellera ensuite `cuda:0`. Vérifiez avec `nvidia-smi` (cellule suivante) qu'il est libre, et ne prenez pas celui d'un collègue.

In [ ]:
import os

GPU_ID = 0          # GPU physique à utiliser : vérifier qu'il est libre avec la cellule suivante
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = os.path.abspath("../cache/")   # même cache Hugging Face que les autres notebooks

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
import copy
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "pas de GPU")

### Paramètres

Quelques explications sur les choix :

- **`STUDENT_MODEL`** : l'encodeur de l'élève. L'enseignant est un DeBERTa-v3 **anglais** appliqué à des textes français. L'élève, lui, doit bien « lire » le français pour retrouver ce que l'enseignant répond : on part donc d'un modèle français, **CamemBERTa-v2** (même architecture que DeBERTa). Alternative à tester ensuite : `microsoft/mdeberta-v3-base` (multilingue).
- **`LAMBDA_FACTEURS`** : poids de la perte sur les 5 dimensions par rapport à la perte sur les 28 émotions (voir partie 6).
- **`LR_ENCODEUR` / `LR_TETE`** : taux d'apprentissage, c'est-à-dire la taille des corrections à chaque étape. L'encodeur est déjà pré-entraîné, on le corrige doucement. La tête part de zéro, on la corrige plus fort.
- **`EVAL_EVERY` / `PATIENCE`** : on évalue sur la validation tous les `EVAL_EVERY` pas, et on s'arrête si aucun progrès n'apparaît après `PATIENCE` évaluations consécutives (*early stopping*).

In [ ]:
# ------------------------------------------------------------- Données
DATA_DIR      = Path("../data")
EMOTIONS_PATH = DATA_DIR / "xp_emotions_28.csv"        # 28 probabilités du zero-shot
DATES_PATH    = DATA_DIR / "idxp_dates.csv"            # date de chaque expérience
VERBATIM_PATH = DATA_DIR / "verbatim_2026_07_31.csv"   # textes (Titre + Description)
ZSCORES_PATH  = DATA_DIR / "Zscores_dimensions.csv"    # 5 scores de Bartlett de la thèse
ID = "id_experience"                                   # clé de jointure commune

# ------------------------------------------------------------- Élève
STUDENT_MODEL = "almanach/camembertav2-base"           # alternative : "microsoft/mdeberta-v3-base"

# ------------------------------------------------------------- Entraînement
SEED            = 42
TEST_FRAC       = 0.10    # les 10 % de verbatims les PLUS RÉCENTS
VAL_FRAC        = 0.10
BATCH_SIZE      = 32
EPOCHS          = 3
LR_ENCODEUR     = 3e-5
LR_TETE         = 1e-3
WEIGHT_DECAY    = 0.01
WARMUP_FRAC     = 0.06
LAMBDA_FACTEURS = 1.0
EVAL_EVERY      = 1000    # pas d'entraînement entre deux évaluations
PATIENCE        = 4

# ------------------------------------------------------------- Production
N_THREADS_CPU = 4         # nombre de cœurs CPU de la machine de production (pour la mesure de vitesse)

OUT_DIR = Path("models/eleve_emotions")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def fixer_graine(seed):
    """Rend l'expérience reproductible : mêmes tirages aléatoires à chaque exécution."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


fixer_graine(SEED)

## 2. Chargement et jointure des données

On lit quatre fichiers qui partagent la même clé, l'ID de l'expérience. La fonction `lire_csv` :
- reconnaît la colonne d'ID même si elle s'appelle « ID expérience » dans un fichier et `id_experience` dans un autre ;
- convertit l'ID en texte, pour qu'un `12345` lu comme nombre dans un fichier et comme texte dans un autre se rejoignent bien ;
- supprime les doublons d'ID en le signalant.

In [ ]:
NOMS_ID = {"id_experience", "id expérience", "id experience", "idxp", "id_xp", "id"}


def lire_csv(path, sep=None):
    kw = dict(sep=sep, encoding="utf-8-sig")
    if sep is None:          # séparateur détecté automatiquement (, ou ;)
        kw["engine"] = "python"
    df = pd.read_csv(path, **kw)
    df = df.rename(columns={c: ID for c in df.columns if c.strip().lower() in NOMS_ID})
    if ID not in df.columns:
        raise KeyError(f"{path.name} : colonne d'ID introuvable. Colonnes : {list(df.columns)}")
    ids = df[ID]
    if pd.api.types.is_float_dtype(ids):      # 12345.0 -> 12345
        ids = ids.astype("Int64")
    df[ID] = ids.astype(str).str.strip()
    n = len(df)
    df = df.drop_duplicates(ID)
    print(f"{path.name:<28} {n:>8} lignes", f"({n - len(df)} doublons d'ID supprimés)" if n > len(df) else "")
    return df


emotions_df = lire_csv(EMOTIONS_PATH)
dates_df    = lire_csv(DATES_PATH)
verbatim_df = lire_csv(VERBATIM_PATH, sep=";")
z_df        = lire_csv(ZSCORES_PATH)

### Vérifications

On contrôle que le fichier d'émotions contient bien **28 colonnes de probabilités entre 0 et 1**, et que le fichier de Z-scores contient les 5 dimensions. Si une assertion échoue, le message indique quoi corriger.

**Point important : le texte.** L'élève doit lire **le même texte que celui que le zero-shot a lu**. On reconstruit ici `Titre + saut de ligne + Description`, comme dans les autres notebooks. Si le zero-shot n'a lu que la description, modifiez `construire_texte`.

In [ ]:
EMOTIONS = [c for c in emotions_df.columns if c != ID and pd.api.types.is_numeric_dtype(emotions_df[c])]
assert len(EMOTIONS) == 28, f"{len(EMOTIONS)} colonnes numériques trouvées au lieu de 28 : {EMOTIONS}"
mini, maxi = emotions_df[EMOTIONS].min().min(), emotions_df[EMOTIONS].max().max()
assert 0 <= mini and maxi <= 1, f"Les émotions doivent être des probabilités entre 0 et 1 (min={mini}, max={maxi})"

DIMENSIONS = [c for c in z_df.columns if c != ID and pd.api.types.is_numeric_dtype(z_df[c])]
assert len(DIMENSIONS) == 5, f"{len(DIMENSIONS)} dimensions trouvées au lieu de 5 : {DIMENSIONS}"

colonnes_date = [c for c in dates_df.columns if c != ID]
COL_DATE = colonnes_date[0]
dates = pd.to_datetime(dates_df[COL_DATE], format="ISO8601", errors="coerce")
if dates.isna().mean() > 0.5:                     # pas au format ISO : on essaie jour/mois/année
    dates = pd.to_datetime(dates_df[COL_DATE], dayfirst=True, errors="coerce")
dates_df["date"] = dates


def construire_texte(frame):
    return (frame["Titre"].fillna("").astype(str).str.strip() + "\n\n"
            + frame["Description"].fillna("").astype(str).str.strip()).str.strip()


verbatim_df["texte"] = construire_texte(verbatim_df)

print("Émotions   :", ", ".join(EMOTIONS))
print("Dimensions :", ", ".join(DIMENSIONS))
print(f"Dates      : colonne {COL_DATE!r}, du {dates.min():%d/%m/%Y} au {dates.max():%d/%m/%Y}, "
      f"{dates.isna().sum()} non lues")

### Jointure

On ne garde que les expériences présentes **dans les quatre fichiers**. Le décompte à chaque étape permet de repérer une perte anormale, par exemple un fichier qui n'a pas le même format d'ID.

In [ ]:
df = emotions_df[[ID] + EMOTIONS]
print(f"{'émotions':<26} {len(df):>8}")
for nom, autre in [("+ textes", verbatim_df[[ID, "texte"]]),
                   ("+ Z-scores", z_df[[ID] + DIMENSIONS]),
                   ("+ dates", dates_df[[ID, "date"]])]:
    df = df.merge(autre, on=ID, how="inner")
    print(f"{nom:<26} {len(df):>8}")

avant = len(df)
df = df.dropna(subset=EMOTIONS + DIMENSIONS)
df = df[df["texte"].str.len() > 0].reset_index(drop=True)
print(f"{'après retrait des vides':<26} {len(df):>8}  ({avant - len(df)} retirés)")
df.head(3)

## 3. Reconstruction de la couche factorielle de la thèse

### Le principe

Dans la thèse, chaque score de Bartlett est obtenu en deux temps :
1. on **standardise** chaque émotion : $z_j = (p_j - \text{moyenne}_j) / \text{écart-type}_j$ ;
2. on fait une **somme pondérée** des 28 émotions standardisées, avec des poids $W$ tirés de l'EFA (saturations et unicités) : $\text{score}_k = \sum_j z_j \, W_{jk}$.

Ces deux opérations sont **linéaires**. Au total, chaque score est donc de la forme :

$$\text{score}_k = \sum_{j=1}^{28} p_j \, A_{jk} + c_k$$

Il suffit de connaître la matrice $A$ (28 × 5) et le vecteur $c$ (5 valeurs).

### Comment on retrouve A et c

Vous avez déjà les 28 probabilités **et** les 5 scores pour 160 000 verbatims. On retrouve $A$ et $c$ par une **régression linéaire** des scores sur les probabilités. Si les scores sont bien des scores de Bartlett calculés sur ces probabilités brutes, la relation est **exacte** : on doit obtenir un **R² de 1,0000** (à l'arrondi près) pour chaque dimension.

C'est un contrôle précieux : il vérifie que le fichier de Z-scores et le fichier d'émotions correspondent exactement. Si un R² est sensiblement inférieur à 1, quelque chose diffère (autre échantillon pour l'EFA, autre transformation des probabilités, Z-scores re-standardisés…), et il faut comprendre pourquoi avant d'aller plus loin.

In [ ]:
P = df[EMOTIONS].to_numpy(np.float64)
Z = df[DIMENSIONS].to_numpy(np.float64)

X = np.column_stack([P, np.ones(len(P))])        # la colonne de 1 permet d'estimer c
coef, *_ = np.linalg.lstsq(X, Z, rcond=None)
A, c = coef[:-1], coef[-1]                        # A : 28 × 5, c : 5

Z_hat = P @ A + c
r2 = 1 - ((Z - Z_hat) ** 2).sum(0) / ((Z - Z.mean(0)) ** 2).sum(0)
display(pd.DataFrame({"R²": r2, "erreur max": np.abs(Z - Z_hat).max(0)}, index=DIMENSIONS).round(6))

if r2.min() < 0.999:
    print("ATTENTION : la relation n'est pas exactement linéaire. "
          "Vérifier que les Z-scores ont été calculés sur ces mêmes probabilités brutes.")
else:
    print("La couche factorielle de la thèse est reconstruite exactement.")

### Lire la matrice A

Chaque colonne indique comment une dimension « pèse » les 28 émotions. Les poids sont exprimés pour des probabilités brutes : une valeur de 3 signifie que passer de 0 à 1 sur cette émotion ajoute 3 points au score. C'est cohérent avec vos saturations : les émotions qui saturent fortement une dimension doivent y avoir les plus gros poids.

In [ ]:
A_df = pd.DataFrame(A, index=EMOTIONS, columns=DIMENSIONS)
fig, ax = plt.subplots(figsize=(6, 9))
lim = np.abs(A).max()
im = ax.imshow(A, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(DIMENSIONS)), DIMENSIONS, rotation=45, ha="right")
ax.set_yticks(range(len(EMOTIONS)), EMOTIONS)
plt.colorbar(im, ax=ax, label="poids dans le score de Bartlett")
ax.set_title("Couche factorielle figée (A)")
plt.tight_layout()
plt.show()

with open(OUT_DIR / "couche_factorielle.json", "w", encoding="utf-8") as f:
    json.dump({"emotions": EMOTIONS, "dimensions": DIMENSIONS, "A": A.tolist(), "c": c.tolist()},
              f, ensure_ascii=False, indent=1)

## 4. Découpage entraînement / validation / test

- **Test = les 10 % les plus récents.** On évalue l'élève sur des verbatims postérieurs à tout ce qu'il a vu, comme ce sera le cas en production. Si les avis évoluent dans le temps, ce test le révélera.
- **Validation = 10 % tirés au hasard parmi le reste.** Elle sert à surveiller l'entraînement et à choisir le meilleur moment pour s'arrêter.
- **Entraînement = tout le reste.**

Le jeu de test n'est utilisé **qu'une seule fois**, à la fin. Si on le consultait pour régler des paramètres, on finirait par ajuster le modèle à ce jeu précis, et la note finale serait trop optimiste.

In [ ]:
datees = df[df["date"].notna()].sort_values("date")
non_datees = df[df["date"].isna()]

n_test = int(len(df) * TEST_FRAC)
test_df = datees.iloc[-n_test:]
reste = pd.concat([datees.iloc[:-n_test], non_datees]).sample(frac=1, random_state=SEED)
n_val = int(len(df) * VAL_FRAC)
val_df = reste.iloc[:n_val]
train_df = reste.iloc[n_val:]

for nom, part in [("entraînement", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{nom:<13} {len(part):>7} verbatims | du {part['date'].min():%d/%m/%Y} au {part['date'].max():%d/%m/%Y}")

# Scores factoriels de l'enseignant, recalculés avec la couche figée (identiques aux Z-scores si R² = 1)
def scores_enseignant(frame):
    return frame[EMOTIONS].to_numpy(np.float64) @ A + c

## 5. Tokenisation et longueur des textes

Un modèle de langue ne lit pas des lettres mais des **tokens** : des morceaux de mots tirés d'un vocabulaire fixe (« administration » peut devenir `▁administr` + `ation`). Le **tokenizer** fait ce découpage et remplace chaque token par son numéro dans le vocabulaire.

Le temps de calcul augmente fortement avec le nombre de tokens. On fixe donc une longueur maximale `MAX_LEN` : au-delà, le texte est tronqué. On la choisit pour couvrir **98 % des verbatims** : les 2 % les plus longs perdent leur fin, mais tous les autres sont lus en entier.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)

exemple = train_df["texte"].iloc[0]
print("Texte  :", exemple[:150].replace("\n", " "), "...")
print("Tokens :", tokenizer.tokenize(exemple)[:25], "...")
print("Numéros:", tokenizer(exemple)["input_ids"][:25], "...")

In [ ]:
echantillon = train_df["texte"].sample(min(10_000, len(train_df)), random_state=SEED).tolist()
longueurs = np.array([len(ids) for ids in tokenizer(echantillon)["input_ids"]])

for q in [50, 90, 95, 98, 99]:
    print(f"{q:>3} % des verbatims font moins de {np.percentile(longueurs, q):>5.0f} tokens")

MAX_LEN = int(min(512, np.ceil(np.percentile(longueurs, 98) / 32) * 32))
print(f"\nMAX_LEN = {MAX_LEN} tokens ({(longueurs > MAX_LEN).mean():.1%} des verbatims seront tronqués)")

plt.figure(figsize=(7, 3))
plt.hist(np.minimum(longueurs, 600), bins=60)
plt.axvline(MAX_LEN, color="red", label=f"MAX_LEN = {MAX_LEN}")
plt.xlabel("nombre de tokens"); plt.ylabel("verbatims"); plt.legend()
plt.show()

### Les données en paquets (*batches*)

Le GPU traite les verbatims par paquets de `BATCH_SIZE`. Dans un paquet, les textes doivent avoir la même longueur : on complète les plus courts avec un token vide (*padding*). Le **masque d'attention** (`attention_mask`) vaut 1 sur les vrais tokens et 0 sur le padding, pour que le modèle ignore ce dernier.

On complète seulement jusqu'au texte le plus long **du paquet** (*padding dynamique*), pas jusqu'à `MAX_LEN` : c'est nettement plus rapide quand la plupart des textes sont courts.

In [ ]:
class VerbatimDataset(Dataset):
    def __init__(self, frame):
        self.textes = frame["texte"].tolist()
        self.cibles = frame[EMOTIONS].to_numpy(np.float32)   # les 28 probabilités de l'enseignant

    def __len__(self):
        return len(self.textes)

    def __getitem__(self, i):
        return self.textes[i], self.cibles[i]


def assembler(batch):
    textes, cibles = zip(*batch)
    enc = tokenizer(list(textes), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
            "cibles": torch.from_numpy(np.stack(cibles))}


train_loader = DataLoader(VerbatimDataset(train_df), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=assembler, num_workers=4, pin_memory=True)

paquet = next(iter(train_loader))
for k, v in paquet.items():
    print(f"{k:<15} {tuple(v.shape)}")

## 6. L'élève

L'élève a trois étages.

**a) L'encodeur (le « lecteur »).** Il transforme chaque token en un vecteur de 768 nombres qui représente son sens *dans le contexte* de la phrase. Il est pré-entraîné sur des milliards de mots de français : il « sait lire », il reste à lui apprendre ce que l'enseignant répond.

**b) Le résumé du texte (*mean pooling*).** On fait la moyenne des vecteurs de tous les vrais tokens (padding exclu grâce au masque). On obtient **un seul vecteur de 768 nombres pour tout le verbatim**.

**c) La tête (le « jury »).** Une couche linéaire transforme ce vecteur en **28 nombres** (*logits*), un par émotion. Une **sigmoïde** ramène chaque nombre entre 0 et 1, **indépendamment des autres** : c'est ce qui garantit l'évaluation indépendante des 28 émotions, comme le zero-shot en multi-label.

**d) La couche factorielle figée.** Elle applique $A$ et $c$ (partie 3) pour obtenir les 5 scores. Elle est enregistrée comme `buffer` : elle fait partie du modèle, mais l'entraînement ne la modifie **jamais**. Elle est calculée en précision complète (float32), car la précision réduite utilisée pour accélérer l'entraînement déformerait légèrement les scores.

**Un détail qui accélère l'entraînement.** On initialise le biais de la tête pour que l'élève non entraîné prédise d'emblée la **probabilité moyenne** de chaque émotion. Il part ainsi d'une réponse raisonnable au lieu de 0,5 partout.

In [ ]:
class Eleve(nn.Module):
    def __init__(self, nom_modele, A, c, n_emotions):
        super().__init__()
        self.encodeur = AutoModel.from_pretrained(nom_modele)
        h = self.encodeur.config.hidden_size
        self.tete = nn.Sequential(nn.Dropout(0.1), nn.Linear(h, n_emotions))
        self.register_buffer("A", torch.tensor(A, dtype=torch.float32))   # figé
        self.register_buffer("c", torch.tensor(c, dtype=torch.float32))   # figé

    def forward(self, input_ids, attention_mask):
        etats = self.encodeur(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        masque = attention_mask.unsqueeze(-1).to(etats.dtype)
        resume = (etats * masque).sum(1) / masque.sum(1).clamp(min=1e-6)       # mean pooling
        logits = self.tete(resume).float()
        with torch.autocast(device_type=logits.device.type, enabled=False):     # précision complète
            probs = torch.sigmoid(logits)
            facteurs = probs @ self.A + self.c
        return logits, probs, facteurs


model = Eleve(STUDENT_MODEL, A, c, len(EMOTIONS)).to(device)

p_moy = train_df[EMOTIONS].mean().clip(1e-4, 1 - 1e-4).to_numpy()
with torch.no_grad():
    model.tete[1].bias.copy_(torch.tensor(np.log(p_moy / (1 - p_moy)), dtype=torch.float32))
    model.tete[1].weight.mul_(0.1)

n_param = sum(p.numel() for p in model.parameters())
print(f"Paramètres de l'élève : {n_param / 1e6:.0f} millions (enseignant : ~435 millions, lu 28 fois)")

### Ce qui se passe à l'intérieur, sur un paquet

Suivons les dimensions des tenseurs à chaque étage, pour le paquet chargé plus haut.

In [ ]:
model.eval()
with torch.no_grad():
    ids, masque = paquet["input_ids"].to(device), paquet["attention_mask"].to(device)
    etats = model.encodeur(input_ids=ids, attention_mask=masque).last_hidden_state
    logits, probs, facteurs = model(ids, masque)
model.train()

print(f"tokens en entrée     {tuple(ids.shape)}        (verbatims × tokens)")
print(f"sortie de l'encodeur {tuple(etats.shape)}   (verbatims × tokens × 768)")
print(f"probabilités         {tuple(probs.shape)}         (verbatims × 28 émotions)")
print(f"scores factoriels    {tuple(facteurs.shape)}          (verbatims × 5 dimensions)")
print("\nPremier verbatim, avant entraînement :")
print(pd.DataFrame({"élève": probs[0].cpu().numpy(), "enseignant": paquet["cibles"][0].numpy()},
                   index=EMOTIONS).round(2).T)

### La fonction de perte : mesurer l'écart avec l'enseignant

À chaque pas, on mesure à quel point l'élève s'écarte de l'enseignant. C'est la **perte** : plus elle est basse, meilleure est l'imitation.

1. **Perte sur les 28 émotions : entropie croisée binaire avec cibles souples.** Pour chaque émotion, on compare la probabilité de l'élève à celle de l'enseignant (0,83, pas « oui »). Cette perte est minimale quand l'élève donne **exactement** la même probabilité. C'est la mesure standard pour des probabilités indépendantes.
2. **Perte sur les 5 dimensions : erreur quadratique moyenne** entre les scores factoriels de l'élève et ceux de l'enseignant. Elle pousse l'élève à soigner surtout les émotions qui pèsent lourd dans vos dimensions.

Perte totale = perte émotions + `LAMBDA_FACTEURS` × perte dimensions.

La perte sur les émotions ne descend jamais à 0 : même un élève parfait garde une valeur plancher, liée à l'incertitude des cibles elles-mêmes (une cible à 0,5 est incertaine par nature). C'est sa **baisse** qui compte, pas sa valeur absolue.

In [ ]:
def calculer_perte(logits, facteurs, cibles):
    perte_emo = F.binary_cross_entropy_with_logits(logits, cibles)
    facteurs_cibles = cibles @ model.A + model.c
    perte_dim = F.mse_loss(facteurs, facteurs_cibles)
    return perte_emo + LAMBDA_FACTEURS * perte_dim, perte_emo, perte_dim

### L'optimiseur et le calendrier du taux d'apprentissage

- **AdamW** ajuste chaque paramètre dans la direction qui fait baisser la perte. Le *weight decay* freine légèrement les poids trop grands, ce qui limite le sur-apprentissage.
- **Deux taux d'apprentissage** : petit pour l'encodeur (on ne veut pas abîmer ce qu'il sait déjà), plus grand pour la tête (elle part de zéro).
- **Échauffement puis décroissance** : le taux monte progressivement pendant les premiers 6 % des pas (les premières corrections, encore très imprécises, ne doivent pas tout bouleverser), puis il décroît linéairement jusqu'à 0.

In [ ]:
sans_decay = ("bias", "LayerNorm.weight", "layernorm.weight", "norm.weight")
groupes = [
    {"params": [p for n, p in model.encodeur.named_parameters() if not any(s in n for s in sans_decay)],
     "lr": LR_ENCODEUR, "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.encodeur.named_parameters() if any(s in n for s in sans_decay)],
     "lr": LR_ENCODEUR, "weight_decay": 0.0},
    {"params": model.tete.parameters(), "lr": LR_TETE, "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(groupes)

n_pas = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_FRAC * n_pas), n_pas)
print(f"{len(train_loader)} pas par epoch, {n_pas} pas au maximum, "
      f"une évaluation tous les {EVAL_EVERY} pas")

### Prédire et mesurer la fidélité

`predire` fait passer un jeu de données dans l'élève, sans calculer de corrections (`torch.no_grad`), et renvoie les 28 probabilités et les 5 scores.

La mesure principale est la **corrélation de Pearson** entre l'élève et l'enseignant, calculée dimension par dimension sur tous les verbatims. Elle vaut 1 si l'élève classe et espace les verbatims exactement comme l'enseignant. Repères pour vos 5 dimensions : **au-dessus de 0,90 c'est bon, au-dessus de 0,95 c'est excellent**.

In [ ]:
@torch.no_grad()
def predire(model, frame, batch_size=128):
    model.eval()
    loader = DataLoader(VerbatimDataset(frame), batch_size=batch_size, shuffle=False,
                        collate_fn=assembler, num_workers=4)
    probs_l, fact_l = [], []
    for b in loader:
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            _, probs, facteurs = model(b["input_ids"].to(device), b["attention_mask"].to(device))
        probs_l.append(probs.float().cpu())
        fact_l.append(facteurs.float().cpu())
    model.train()
    return torch.cat(probs_l).numpy(), torch.cat(fact_l).numpy()


def correlations(a, b):
    """Corrélation de Pearson colonne par colonne entre deux matrices de même forme."""
    a = a - a.mean(0)
    b = b - b.mean(0)
    return (a * b).sum(0) / np.sqrt((a ** 2).sum(0) * (b ** 2).sum(0))


P_val = val_df[EMOTIONS].to_numpy()
Z_val = scores_enseignant(val_df)

**Point de départ.** Avant tout entraînement, l'élève prédit à peu près la moyenne pour tout le monde : les corrélations doivent être proches de 0. C'est la référence à partir de laquelle on mesure les progrès.

In [ ]:
Pv, Fv = predire(model, val_df)
print("Corrélations avant entraînement :")
print(pd.Series(correlations(Fv, Z_val), index=DIMENSIONS).round(3).to_string())

## 7. Entraînement

À chaque pas :
1. on charge un paquet de 32 verbatims sur le GPU ;
2. l'élève prédit (*forward*), en précision réduite **bfloat16** pour aller environ deux fois plus vite ;
3. on calcule la perte ;
4. on calcule dans quelle direction modifier chaque paramètre pour la réduire (*backward*) ;
5. on limite la taille de la correction (*gradient clipping*), par sécurité contre les pas aberrants ;
6. l'optimiseur applique la correction.

Tous les `EVAL_EVERY` pas, on mesure la fidélité sur la validation. **Si elle s'améliore, on sauvegarde le modèle.** Sinon, on compte un échec, et on arrête après `PATIENCE` échecs consécutifs.

Comptez de l'ordre de 10 à 30 minutes par epoch sur une H100. La barre de progression affiche la perte moyenne récente : elle doit baisser rapidement au début, puis plus lentement.

**Si le noyau plante** : le meilleur modèle est déjà sauvegardé dans `models/eleve_emotions/meilleur.pt`. Vous pouvez passer directement à la partie 8.

In [ ]:
historique = []
meilleur_score, echecs, pas, arret = -np.inf, 0, 0, False
cumul = {"totale": 0.0, "émotions": 0.0, "dimensions": 0.0, "n": 0}
debut = time.time()

model.train()
for epoch in range(EPOCHS):
    barre = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
    for batch in barre:
        ids = batch["input_ids"].to(device, non_blocking=True)
        masque = batch["attention_mask"].to(device, non_blocking=True)
        cibles = batch["cibles"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            logits, probs, facteurs = model(ids, masque)
        perte, perte_emo, perte_dim = calculer_perte(logits, facteurs, cibles)

        optimizer.zero_grad(set_to_none=True)
        perte.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        pas += 1

        cumul["totale"] += perte.item(); cumul["émotions"] += perte_emo.item()
        cumul["dimensions"] += perte_dim.item(); cumul["n"] += 1
        if pas % 50 == 0:
            barre.set_postfix(perte=f"{cumul['totale'] / cumul['n']:.4f}")

        if pas % EVAL_EVERY == 0 or pas == n_pas:
            Pv, Fv = predire(model, val_df)
            r_dim = correlations(Fv, Z_val)
            r_emo = correlations(Pv, P_val)
            ligne = {"pas": pas, "epoch": epoch + 1,
                     "perte_totale": cumul["totale"] / cumul["n"],
                     "perte_emotions": cumul["émotions"] / cumul["n"],
                     "perte_dimensions": cumul["dimensions"] / cumul["n"],
                     "r_dimensions_moyen": r_dim.mean(), "r_emotions_moyen": r_emo.mean(),
                     **{f"r_{d}": r for d, r in zip(DIMENSIONS, r_dim)}}
            historique.append(ligne)
            cumul = {k: 0.0 for k in cumul} | {"n": 0}

            if r_dim.mean() > meilleur_score:
                meilleur_score, echecs = r_dim.mean(), 0
                torch.save(model.state_dict(), OUT_DIR / "meilleur.pt")
                statut = "nouveau meilleur, sauvegardé"
            else:
                echecs += 1
                statut = f"pas de progrès ({echecs}/{PATIENCE})"
            print(f"pas {pas:>6} | perte {ligne['perte_totale']:.4f} | r dimensions {r_dim.mean():.4f} "
                  f"| r émotions {r_emo.mean():.4f} | {statut} | {(time.time() - debut) / 60:.0f} min")
            pd.DataFrame(historique).to_csv(OUT_DIR / "historique.csv", index=False)

            if echecs >= PATIENCE:
                arret = True
                break
    if arret:
        print("Arrêt anticipé : la validation ne progresse plus.")
        break

print(f"Terminé en {(time.time() - debut) / 60:.0f} min. Meilleure corrélation moyenne : {meilleur_score:.4f}")

### Lire les courbes

- **À gauche, la perte d'entraînement** doit baisser vite puis se stabiliser.
- **À droite, la fidélité sur la validation** doit monter puis plafonner.
- **Signal de sur-apprentissage** : la perte d'entraînement continue de baisser alors que la validation stagne ou redescend. L'élève apprend alors par cœur les exemples d'entraînement au lieu d'apprendre la règle. L'arrêt anticipé et la sauvegarde du meilleur modèle vous protègent de ce cas.

In [ ]:
h = pd.DataFrame(historique) if historique else pd.read_csv(OUT_DIR / "historique.csv")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(h["pas"], h["perte_emotions"], label="émotions (entropie croisée)")
ax1.plot(h["pas"], h["perte_dimensions"], label="dimensions (erreur quadratique)")
ax1.set_xlabel("pas"); ax1.set_title("Perte d'entraînement"); ax1.legend()
for d in DIMENSIONS:
    ax2.plot(h["pas"], h[f"r_{d}"], label=d)
ax2.axhline(0.95, color="grey", ls="--", lw=0.8)
ax2.set_xlabel("pas"); ax2.set_title("Corrélation élève / enseignant (validation)"); ax2.legend()
plt.tight_layout()
plt.show()

## 8. Évaluation finale sur le test (les verbatims les plus récents)

On recharge le **meilleur** modèle sauvegardé (pas forcément le dernier), puis on mesure la fidélité à quatre niveaux.

In [ ]:
model.load_state_dict(torch.load(OUT_DIR / "meilleur.pt", map_location=device))
P_test_eleve, F_test_eleve = predire(model, test_df)
P_test_ens = test_df[EMOTIONS].to_numpy()
F_test_ens = scores_enseignant(test_df)

### Niveau 1 : les 5 dimensions (l'indicateur principal)

- **r** : corrélation de Pearson. Visez au-dessus de 0,90, idéalement 0,95.
- **MAE** : écart absolu moyen, dans l'unité des Z-scores. Une MAE de 0,15 signifie qu'en moyenne l'élève se trompe de 0,15 écart-type.
- **MAE relative** : MAE divisée par l'écart-type de la dimension. C'est la plus parlante pour comparer les dimensions entre elles.

In [ ]:
mae = np.abs(F_test_eleve - F_test_ens).mean(0)
res_dim = pd.DataFrame({
    "r": correlations(F_test_eleve, F_test_ens),
    "MAE": mae,
    "MAE relative": mae / F_test_ens.std(0),
}, index=DIMENSIONS)
display(res_dim.round(3))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for k, (ax, d) in enumerate(zip(axes, DIMENSIONS)):
    ax.hexbin(F_test_ens[:, k], F_test_eleve[:, k], gridsize=50, bins="log", cmap="Blues")
    lo, hi = np.percentile(F_test_ens[:, k], [0.5, 99.5])
    ax.plot([lo, hi], [lo, hi], "r--", lw=1)
    ax.set_title(f"{d}  (r = {res_dim.loc[d, 'r']:.3f})")
    ax.set_xlabel("enseignant (thèse)")
axes[0].set_ylabel("élève")
plt.tight_layout()
plt.show()

Sur ces nuages de points, chaque verbatim est un point : plus ils se serrent autour de la diagonale rouge, plus l'imitation est fidèle. Regardez surtout **les extrémités** : un élève peut être bon en moyenne mais écraser les cas extrêmes (les points s'aplatissent à l'horizontale aux bords). Ce sont souvent les verbatims qui vous intéressent le plus, par exemple une détresse forte.

### Niveau 2 : les 28 émotions

Une émotion mal reproduite n'est grave que si elle pèse lourd dans une dimension. Le tableau croise donc la corrélation de chaque émotion avec son poids maximal dans la couche factorielle.

In [ ]:
res_emo = pd.DataFrame({
    "r": correlations(P_test_eleve, P_test_ens),
    "MAE": np.abs(P_test_eleve - P_test_ens).mean(0),
    "poids max |A|": np.abs(A).max(1),
    "dimension principale": [DIMENSIONS[k] for k in np.abs(A).argmax(1)],
}, index=EMOTIONS).sort_values("r")
display(res_emo.round(3))

### Niveau 3 : la structure du modèle factoriel

Avoir de bonnes corrélations dimension par dimension ne garantit pas que l'élève respecte **la structure** de votre modèle, c'est-à-dire le lien entre chaque émotion et chaque dimension.

Pour le vérifier, on calcule pour l'enseignant puis pour l'élève la **matrice de structure** : la corrélation entre chacune des 28 émotions et chacune des 5 dimensions. On compare ensuite les deux matrices, colonne par colonne, avec le **coefficient de congruence de Tucker** (φ). C'est la mesure classique pour comparer deux solutions factorielles : **φ ≥ 0,95** signifie que les deux facteurs sont considérés comme identiques, et entre 0,85 et 0,94 qu'ils sont proches.

On n'a pas besoin de refaire une EFA (avec ses problèmes de rotation et d'ordre des facteurs) : on compare directement les liens émotion-dimension.

In [ ]:
def matrice_structure(P, Fz):
    Pc = (P - P.mean(0)) / P.std(0)
    Fc = (Fz - Fz.mean(0)) / Fz.std(0)
    return Pc.T @ Fc / len(P)            # 28 × 5


S_ens = matrice_structure(P_test_ens, F_test_ens)
S_eleve = matrice_structure(P_test_eleve, F_test_eleve)
phi = (S_ens * S_eleve).sum(0) / np.sqrt((S_ens ** 2).sum(0) * (S_eleve ** 2).sum(0))
res_dim["φ de Tucker"] = phi
display(pd.Series(phi, index=DIMENSIONS, name="φ de Tucker").round(4).to_frame())

### Niveau 4 : la dimension dominante (si vous étiquetez chaque verbatim)

Si vous attribuez à chaque verbatim sa dimension la plus élevée, on mesure le pourcentage de verbatims où l'élève et l'enseignant désignent **la même**. Le **κ de Cohen** corrige ce pourcentage de l'accord dû au hasard.

Rappel : les scores de Bartlett sont centrés sur la moyenne du corpus. « Dimension la plus élevée » veut dire « la plus au-dessus de la moyenne », pas forcément « présente ». Les désaccords se concentrent sur les verbatims où deux dimensions sont presque à égalité, ce qui est normal.

In [ ]:
dom_ens = F_test_ens.argmax(1)
dom_eleve = F_test_eleve.argmax(1)
print(f"Même dimension dominante : {(dom_ens == dom_eleve).mean():.1%}")
print(f"κ de Cohen               : {cohen_kappa_score(dom_ens, dom_eleve):.3f}")

confusion = pd.crosstab(pd.Series(np.array(DIMENSIONS)[dom_ens], name="enseignant"),
                        pd.Series(np.array(DIMENSIONS)[dom_eleve], name="élève"))
display(confusion)

res_dim.to_csv(OUT_DIR / "resultats_test_dimensions.csv")
res_emo.to_csv(OUT_DIR / "resultats_test_emotions.csv")

## 9. Export pour le CPU et mesure de vitesse

Jusqu'ici l'élève tourne en PyTorch sur GPU. Pour la production sur CPU, on fait deux transformations.

**ONNX.** On « fige » le modèle dans un format standard, optimisé pour l'exécution seule : plus besoin de PyTorch en production, seulement `onnxruntime`, beaucoup plus léger. Les réponses ne changent pas, on le vérifie.

**Quantification int8.** Les poids du modèle sont stockés en nombres à virgule de 32 bits. On les convertit en entiers de 8 bits : le modèle devient environ 4 fois plus léger et 2 à 3 fois plus rapide sur CPU, au prix d'une petite perte de précision. **On mesure cette perte** : si la fidélité à l'enseignant baisse trop, on garde la version 32 bits.

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

model_cpu = copy.deepcopy(model).to("cpu").float().eval()


class PourExport(nn.Module):
    """Version du modèle qui ne renvoie que ce dont la production a besoin."""
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        _, probs, facteurs = self.m(input_ids, attention_mask)
        return probs, facteurs


onnx_path = OUT_DIR / "eleve.onnx"
int8_path = OUT_DIR / "eleve_int8.onnx"
exemple_enc = tokenizer(test_df["texte"].iloc[:2].tolist(), truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="pt")
args = (exemple_enc["input_ids"], exemple_enc["attention_mask"])
options = dict(input_names=["input_ids", "attention_mask"], output_names=["probs", "facteurs"],
               dynamic_axes={"input_ids": {0: "lot", 1: "tokens"}, "attention_mask": {0: "lot", 1: "tokens"},
                             "probs": {0: "lot"}, "facteurs": {0: "lot"}},
               opset_version=17)
try:
    torch.onnx.export(PourExport(model_cpu), args, str(onnx_path), dynamo=False, **options)
except TypeError:                     # anciennes versions de torch sans l'argument dynamo
    torch.onnx.export(PourExport(model_cpu), args, str(onnx_path), **options)

quantize_dynamic(str(onnx_path), str(int8_path), weight_type=QuantType.QInt8, per_channel=True)
for p in (onnx_path, int8_path):
    print(f"{p.name:<18} {p.stat().st_size / 1e6:>6.0f} Mo")

### Vérifier que l'export ne change rien, puis mesurer l'effet de l'int8

On fait passer 2 000 verbatims du test dans les deux versions ONNX et on compare à l'enseignant. La version 32 bits doit donner **les mêmes** corrélations que PyTorch ; la version int8, des corrélations à peine plus basses (une perte de 0,005 est acceptable, au-delà de 0,02 il vaut mieux garder la version 32 bits).

In [ ]:
def session_cpu(path, n_threads=N_THREADS_CPU):
    so = ort.SessionOptions()
    so.intra_op_num_threads = n_threads
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])


def predire_onnx(session, textes, batch_size=32):
    probs_l, fact_l = [], []
    for i in range(0, len(textes), batch_size):
        enc = tokenizer(textes[i:i + batch_size], truncation=True, max_length=MAX_LEN,
                        padding=True, return_tensors="np")
        probs, facteurs = session.run(None, {"input_ids": enc["input_ids"].astype(np.int64),
                                             "attention_mask": enc["attention_mask"].astype(np.int64)})
        probs_l.append(probs); fact_l.append(facteurs)
    return np.concatenate(probs_l), np.concatenate(fact_l)


sous_test = test_df.sample(min(2000, len(test_df)), random_state=SEED)
textes_st = sous_test["texte"].tolist()
F_st_ens = scores_enseignant(sous_test)
F_st_torch = F_test_eleve[test_df.index.get_indexer(sous_test.index)]

comparaison = {"PyTorch (GPU)": correlations(F_st_torch, F_st_ens)}
for nom, path in [("ONNX 32 bits", onnx_path), ("ONNX int8", int8_path)]:
    _, F_onnx = predire_onnx(session_cpu(path, n_threads=os.cpu_count()), textes_st)
    comparaison[nom] = correlations(F_onnx, F_st_ens)
display(pd.DataFrame(comparaison, index=DIMENSIONS).round(4))

### Mesurer la vitesse sur CPU

On mesure le temps pour traiter **un verbatim à la fois**, comme en production quand ils arrivent au fil de l'eau, en limitant le calcul à `N_THREADS_CPU` cœurs.

**Attention :** le processeur du serveur GPU n'est pas celui de la machine de production. Les temps absolus seront différents, mais le rapport entre les versions reste indicatif. Refaites cette mesure sur la machine de production avec la partie 10.

In [ ]:
textes_mesure = test_df["texte"].sample(200, random_state=SEED).tolist()


def chronometrer(fonction, textes):
    for t in textes[:5]:           # échauffement, non compté
        fonction(t)
    durees = []
    for t in textes:
        t0 = time.perf_counter()
        fonction(t)
        durees.append(time.perf_counter() - t0)
    return np.array(durees) * 1000


torch.set_num_threads(N_THREADS_CPU)


@torch.no_grad()
def un_texte_torch(t):
    enc = tokenizer([t], truncation=True, max_length=MAX_LEN, return_tensors="pt")
    model_cpu(enc["input_ids"], enc["attention_mask"])


vitesses = {"PyTorch CPU 32 bits": chronometrer(un_texte_torch, textes_mesure)}
for nom, path in [("ONNX 32 bits", onnx_path), ("ONNX int8", int8_path)]:
    s = session_cpu(path)
    vitesses[nom] = chronometrer(lambda t: predire_onnx(s, [t]), textes_mesure)

display(pd.DataFrame({nom: {"moyenne (ms)": d.mean(), "médiane (ms)": np.median(d),
                            "95e centile (ms)": np.percentile(d, 95)}
                      for nom, d in vitesses.items()}).T.round(0))

## 10. En production

Le dossier `models/eleve_emotions/` contient tout ce qu'il faut :

| Fichier | Rôle |
|---|---|
| `eleve_int8.onnx` (ou `eleve.onnx`) | le modèle |
| `tokenizer/` | le découpage en tokens |
| `config_production.json` | noms des émotions et dimensions, `MAX_LEN`, couche factorielle |
| `meilleur.pt` | poids PyTorch, pour ré-entraîner ou ré-exporter plus tard |

Sur la machine de production, il suffit d'installer `onnxruntime`, `transformers` et `numpy` (pas besoin de `torch`), de copier ce dossier, et d'utiliser la fonction `predire_verbatims` ci-dessous.

**Gardez les 28 probabilités en base**, pas seulement les 5 scores : elles permettent d'expliquer chaque résultat, et de vérifier plus tard que la structure factorielle tient toujours.

**Surveillance de la dérive.** Tous les trimestres, faites annoter quelques centaines de nouveaux verbatims par le zero-shot (sur GPU) et comparez-les à l'élève avec les mesures de la partie 8. Si la fidélité baisse, ré-entraînez l'élève en ajoutant ces nouveaux exemples.

In [ ]:
tokenizer.save_pretrained(OUT_DIR / "tokenizer")
config_prod = {"student_model": STUDENT_MODEL, "max_len": MAX_LEN,
               "emotions": EMOTIONS, "dimensions": DIMENSIONS,
               "A": A.tolist(), "c": c.tolist(),
               "resultats_test": res_dim.round(4).to_dict()}
with open(OUT_DIR / "config_production.json", "w", encoding="utf-8") as f:
    json.dump(config_prod, f, ensure_ascii=False, indent=1)
print("Fichiers prêts dans", OUT_DIR.resolve())

### La fonction de production

Cette cellule est **autonome** : elle ne dépend de rien d'autre dans le notebook. Vous pouvez la copier telle quelle dans un script sur la machine de production. Elle renvoie un tableau avec les 28 probabilités et les 5 scores de Bartlett pour chaque verbatim.

In [ ]:
import json
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
from transformers import AutoTokenizer

DOSSIER_MODELE = Path("models/eleve_emotions")
N_THREADS = 4

with open(DOSSIER_MODELE / "config_production.json", encoding="utf-8") as f:
    CONFIG = json.load(f)
TOKENIZER = AutoTokenizer.from_pretrained(DOSSIER_MODELE / "tokenizer")
_so = ort.SessionOptions()
_so.intra_op_num_threads = N_THREADS
SESSION = ort.InferenceSession(str(DOSSIER_MODELE / "eleve_int8.onnx"), _so,
                               providers=["CPUExecutionProvider"])


def predire_verbatims(textes):
    """textes : liste de chaînes (Titre + "\n\n" + Description). Renvoie un DataFrame 28 + 5 colonnes."""
    enc = TOKENIZER(list(textes), truncation=True, max_length=CONFIG["max_len"],
                    padding=True, return_tensors="np")
    probs, facteurs = SESSION.run(None, {"input_ids": enc["input_ids"].astype(np.int64),
                                         "attention_mask": enc["attention_mask"].astype(np.int64)})
    return pd.concat([pd.DataFrame(probs, columns=CONFIG["emotions"]),
                      pd.DataFrame(facteurs, columns=CONFIG["dimensions"])], axis=1)


resultat = predire_verbatims([
    "Démarche simple\n\nTout s'est très bien passé, dossier traité en deux jours.",
    "Inadmissible\n\nTrois mois sans réponse, personne au téléphone, je ne sais plus quoi faire.",
])
resultat[CONFIG["dimensions"]].round(2)